Check the GPU and import libraries

In [ ]:
!pip -q install -U transformers datasets accelerate peft bitsandbytes trl evaluate sentencepiece

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Enable a GPU runtime before running training experiments.")

PyTorch: 2.11.0+cpu
CUDA available: False
Enable a GPU runtime before running training experiments.


Load the pretrained LLM

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

if torch.cuda.is_available():
    model = model.to("cuda")

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded:", MODEL_NAME)
print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-0.5B
Total parameters: 494032768


Test the original model

In [ ]:
def generate_text(prompt, max_new_tokens=80):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

print(generate_text("Explain machine learning in simple words."))

Explain machine learning in simple words. Machine learning is a branch of artificial intelligence that involves training algorithms to make predictions or decisions based on data. It uses algorithms to analyze data and learn from it, allowing machines to improve their performance over time. Machine learning algorithms can be used for a wide range of applications, including image recognition, natural language processing, and predictive analytics.


Prepare a small training dataset

In [ ]:
from datasets import Dataset

examples = [
    {
        "instruction": "What is artificial intelligence?",
        "response": "Artificial intelligence is the field of building computer systems that perform tasks requiring abilities such as learning, reasoning, and understanding language."
    },
    {
        "instruction": "What is machine learning?",
        "response": "Machine learning is a branch of AI in which models learn patterns from data to make predictions or decisions."
    },
    {
        "instruction": "What is deep learning?",
        "response": "Deep learning uses neural networks with multiple layers to learn complex patterns from data."
    },
    {
        "instruction": "What is an LLM?",
        "response": "A large language model is a neural network trained on text to predict and generate sequences of tokens."
    },
    {
        "instruction": "What is NLP?",
        "response": "Natural language processing is the field of AI that develops methods for computers to process and generate human language."
    },
    {
        "instruction": "Explain a transformer.",
        "response": "A transformer is a neural network architecture that uses attention mechanisms to model relationships between tokens."
    },
    {
        "instruction": "What is an embedding?",
        "response": "An embedding is a numerical vector representation of an item such as a word, token, or document."
    },
    {
        "instruction": "What is fine-tuning?",
        "response": "Fine-tuning continues training a pretrained model on a task-specific dataset to adapt its behavior."
    },
    {
        "instruction": "What is a neural network?",
        "response": "A neural network is a model made of connected computational units that learn patterns by adjusting their parameters."
    },
    {
        "instruction": "What is a token?",
        "response": "A token is a unit of text processed by a language model. It may represent a word, part of a word, or a symbol."
    },
]

dataset = Dataset.from_list(examples)
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=SEED
)

train_data = dataset["train"]
eval_data = dataset["test"]

print("Training examples:", len(train_data))
print("Evaluation examples:", len(eval_data))
print(train_data[0])

Training examples: 8
Evaluation examples: 2
{'instruction': 'What is artificial intelligence?', 'response': 'Artificial intelligence is the field of building computer systems that perform tasks requiring abilities such as learning, reasoning, and understanding language.'}


Format and tokenize the data

In [ ]:
from transformers import DataCollatorForLanguageModeling

MAX_LENGTH = 256

def format_example(example):
    return (
        "### Question:\n"
        + example["instruction"]
        + "\n\n### Answer:\n"
        + example["response"]
        + tokenizer.eos_token
    )

def tokenize_example(example):
    return tokenizer(
        format_example(example),
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_data.map(
    tokenize_example,
    remove_columns=train_data.column_names,
)

tokenized_eval = eval_data.map(
    tokenize_example,
    remove_columns=eval_data.column_names,
)

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Train the model

In [ ]:
from transformers import Trainer, TrainingArguments

sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

args = TrainingArguments(
    output_dir="/content/sft_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

sft_trainer = Trainer(
    model=sft_model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=collator,
)

sft_trainer.train()
sft_trainer.save_model("/content/sft_model/final")
tokenizer.save_pretrained("/content/sft_model/final")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.020116,2.304174
2,0.926964,2.201824
3,0.852694,2.182866


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/sft_model/final/tokenizer_config.json',
 '/content/sft_model/final/chat_template.jinja',
 '/content/sft_model/final/tokenizer.json')

Evaluate the SFT model

In [ ]:
sft_model = sft_trainer.model
sft_model.eval()

def generate_with_model(model, prompt, max_new_tokens=80):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

prompt = "### Question:\nWhat is an LLM?\n\n### Answer:\n"
print(generate_with_model(sft_model, prompt))

### Question:
What is an LLM?

### Answer:
An LLM is a large language model. It is a model that can generate text. It is a model that is trained on a large amount of text data. It is trained on a large amount of text data. It is trained on a large amount of text data. It is trained on a large amount of text data. It is trained on a large amount of text data. It is trained


Create an instruction-following dataset

In [ ]:
instruction_examples = [
    {
        "instruction": "Define NLP in one sentence.",
        "response": "NLP enables computers to process, analyze, and generate human language."
    },
    {
        "instruction": "Give three applications of AI.",
        "response": "Three applications of AI are medical image analysis, language translation, and fraud detection."
    },
    {
        "instruction": "Explain embeddings to a beginner.",
        "response": "Embeddings represent words or other items as numerical vectors that a machine learning model can process."
    },
    {
        "instruction": "Summarize machine learning in 15 words.",
        "response": "Machine learning helps computers discover patterns in data and use them to make predictions or decisions."
    },
    {
        "instruction": "List two common NLP tasks.",
        "response": "Two common NLP tasks are sentiment analysis and named entity recognition."
    },
    {
        "instruction": "Explain the difference between AI and ML.",
        "response": "AI is the broader field of intelligent computer systems, while ML is an approach that learns patterns from data."
    },
]

instruction_dataset = Dataset.from_list(instruction_examples)

def format_instruction(example):
    return (
        "Follow the instruction carefully.\n"
        f"Instruction: {example['instruction']}\n"
        f"Response: {example['response']}"
        + tokenizer.eos_token
    )

def tokenize_instruction(example):
    return tokenizer(
        format_instruction(example),
        truncation=True,
        max_length=MAX_LENGTH,
    )

instruction_tokens = instruction_dataset.map(
    tokenize_instruction,
    remove_columns=instruction_dataset.column_names,
)

print(instruction_tokens[0].keys())

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

dict_keys(['input_ids', 'attention_mask'])


Train an instruction-tuned model

In [ ]:
instruction_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

instruction_args = TrainingArguments(
    output_dir="/content/instruction_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

instruction_trainer = Trainer(
    model=instruction_model,
    args=instruction_args,
    train_dataset=instruction_tokens,
    data_collator=collator,
)

instruction_trainer.train()
instruction_trainer.save_model(
    "/content/instruction_model/final"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,2.580038
2,1.891256
3,0.983025
4,0.798651
5,0.615390
6,0.643371


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Compare trainable parameters

In [ ]:
from peft import (
    LoraConfig,
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)

peft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

peft_model = get_peft_model(
    peft_model,
    lora_config
)

peft_model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

trainable params: 294,912 || all params: 125,534,208 || trainable%: 0.2349


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 40.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 44.7 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers]


Train LoRA adapters

In [ ]:
from transformers import Trainer, TrainingArguments

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

lora_model = get_peft_model(
    lora_model,
    lora_config
)

lora_args = TrainingArguments(
    output_dir="/content/lora_model",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=tokenized_train,
    data_collator=collator,
)

lora_model.save_pretrained("/content/lora_adapter")
tokenizer.save_pretrained("/content/lora_adapter")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

('/content/lora_adapter/tokenizer_config.json',
 '/content/lora_adapter/chat_template.jinja',
 '/content/lora_adapter/tokenizer.json')

Reload the saved adapter

In [ ]:
from peft import PeftModel

base_for_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

reloaded_lora = PeftModel.from_pretrained(
    base_for_lora,
    "/content/lora_adapter"
)

print(
    generate_with_model(
        reloaded_lora,
        "### Question:\nWhat is machine learning?\n\n### Answer:"
    )
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

### Question:
What is machine learning?

### Answer:               *&###' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning' learning


Prompt Tuning

In [ ]:
from peft import (
    PromptTuningConfig,
    PromptTuningInit,
    TaskType,
    get_peft_model,
)
from transformers import AutoModelForCausalLM

prompt_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

prompt_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="Answer questions clearly and accurately.",
    tokenizer_name_or_path=MODEL_NAME,
)

prompt_model = get_peft_model(
    prompt_model,
    prompt_config
)

prompt_model.print_trainable_parameters()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  251MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  251MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

trainable params: 6,144 || all params: 125,245,440 || trainable%: 0.0049


Train the learned prompt